Random Forest – Classification & Regression
Dataset: `Breast Cancer` (classification) · `California Housing` (regression)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay,
                             mean_squared_error, mean_absolute_error, r2_score)
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded ✅")


Part 1 — Classification (Breast Cancer Dataset)

In [ ]:
bc = load_breast_cancer()
X, y = bc.data, bc.target
print(f"Features : {bc.feature_names[:5]} ...")
print(f"Classes  : {bc.target_names}")
print(f"Shape    : {X.shape}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")


Baseline Random Forest

In [ ]:
rf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)

print("Baseline Accuracy :", accuracy_score(y_test, y_pred_base))
print("\nClassification Report:\n", classification_report(y_test, y_pred_base,
      target_names=bc.target_names))


Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
param_dist_clf = {
    'n_estimators'     : [50, 100, 200, 300],
    'max_depth'        : [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2'],
    'bootstrap'        : [True, False]
}

rand_clf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist_clf, n_iter=20, cv=5,
    scoring='accuracy', random_state=42, n_jobs=-1)

rand_clf.fit(X_train, y_train)
print("Best Params :", rand_clf.best_params_)
print("Best CV Acc :", round(rand_clf.best_score_, 4))


In [ ]:
best_rf_clf = rand_clf.best_estimator_
y_pred_best = best_rf_clf.predict(X_test)

print("Tuned Accuracy :", accuracy_score(y_test, y_pred_best))
print("\nClassification Report:\n", classification_report(y_test, y_pred_best,
      target_names=bc.target_names))


Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=bc.target_names)
disp.plot(cmap='Greens')
plt.title("Random Forest – Confusion Matrix (Classification)")
plt.tight_layout()
plt.show()


Top Feature Importances

In [ ]:
importances = best_rf_clf.feature_importances_
idx = np.argsort(importances)[::-1][:15]  # top 15

plt.figure(figsize=(10, 4))
plt.bar(range(len(idx)), importances[idx], color='seagreen')
plt.xticks(range(len(idx)), [bc.feature_names[i] for i in idx], rotation=45, ha='right', fontsize=8)
plt.title("Random Forest – Top-15 Feature Importances (Classification)")
plt.tight_layout()
plt.show()



Regression (California Housing Dataset)

In [ ]:
housing = fetch_california_housing()
X_r, y_r = housing.data, housing.target
print(f"Features: {housing.feature_names}")
print(f"Shape   : {X_r.shape}")


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_r, y_r, test_size=0.2, random_state=42)
print(f"Train: {X_tr.shape}  Test: {X_te.shape}")


Baseline Random Forest Regressor

In [ ]:
rf_reg_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg_base.fit(X_tr, y_tr)
y_pred_reg_base = rf_reg_base.predict(X_te)

rmse = np.sqrt(mean_squared_error(y_te, y_pred_reg_base))
print(f"Baseline  RMSE : {rmse:.4f}")
print(f"Baseline  MAE  : {mean_absolute_error(y_te, y_pred_reg_base):.4f}")
print(f"Baseline  R²   : {r2_score(y_te, y_pred_reg_base):.4f}")


Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
param_dist_reg = {
    'n_estimators'     : [50, 100, 200],
    'max_depth'        : [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2', 0.5]
}

rand_reg = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_dist_reg, n_iter=15, cv=5,
    scoring='neg_root_mean_squared_error', random_state=42, n_jobs=-1)

rand_reg.fit(X_tr, y_tr)
print("Best Params :", rand_reg.best_params_)
print("Best CV RMSE:", round(-rand_reg.best_score_, 4))


In [ ]:
best_rf_reg = rand_reg.best_estimator_
y_pred_reg_best = best_rf_reg.predict(X_te)

rmse_t = np.sqrt(mean_squared_error(y_te, y_pred_reg_best))
print(f"Tuned  RMSE : {rmse_t:.4f}")
print(f"Tuned  MAE  : {mean_absolute_error(y_te, y_pred_reg_best):.4f}")
print(f"Tuned  R²   : {r2_score(y_te, y_pred_reg_best):.4f}")


Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_te, y_pred_reg_best, alpha=0.2, s=8, color='seagreen')
plt.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', lw=2)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Random Forest – Actual vs Predicted (Regression)")
plt.tight_layout()
plt.show()


Feature Importances

In [ ]:
importances_r = best_rf_reg.feature_importances_
feat_names    = housing.feature_names
idx_r = np.argsort(importances_r)[::-1]

plt.figure(figsize=(8, 4))
plt.bar(range(len(idx_r)), importances_r[idx_r], color='seagreen')
plt.xticks(range(len(idx_r)), [feat_names[i] for i in idx_r], rotation=45, ha='right')
plt.title("Random Forest – Feature Importances (Regression)")
plt.tight_layout()
plt.show()


OOB Score (Out-of-Bag)

In [ ]:
rf_oob = RandomForestRegressor(**{**rand_reg.best_params_,
                                  'oob_score': True, 'random_state': 42, 'n_jobs': -1})
rf_oob.fit(X_tr, y_tr)
print(f"OOB R² Score: {rf_oob.oob_score_:.4f}")
